# PCA Dataset Classification with MLP
## Multi-Layer Perceptron (Neural Network) Analysis
Training and evaluating MLP on PCA-transformed dataset with train/validation/test split.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import joblib
import os

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load and Explore Dataset

In [ ]:
# Load dataset (PCA)
df = pd.read_csv('../datasets/trojan_cleaned(pca).csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

## 2. Data Preparation

In [ ]:
# Separate features and target
# Assuming the last column is the target (adjust if needed)
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

## 3. Train/Validation/Test Split

In [ ]:
# Splitting Dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    X, 
    y, 
    test_size=0.30,
    random_state=42
)

# Splitting the 30% --> 2 x 15% (test and val)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=0.50,
    random_state=42
)

print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Validation set size: {X_val.shape[0]} ({X_val.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/X.shape[0]*100:.1f}%)")
print(f"\nTotal: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} samples")

## 4. MLP (Multi-Layer Perceptron) Model

In [ ]:
# Train MLP (Multi-Layer Perceptron)
print("\n" + "="*50)
print("MULTI-LAYER PERCEPTRON (MLP) - Neural Network")
print("="*50)

# MLP with optimized architecture for PCA data
# Hidden layers: [100, 50] neurons with ReLU activation
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)
mlp_model.fit(X_train, y_train)

# Predictions
mlp_train_pred = mlp_model.predict(X_train)
mlp_val_pred = mlp_model.predict(X_val)
mlp_test_pred = mlp_model.predict(X_test)

# Probabilities for ROC curve
mlp_train_proba = mlp_model.predict_proba(X_train)[:, 1]
mlp_val_proba = mlp_model.predict_proba(X_val)[:, 1]
mlp_test_proba = mlp_model.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("\nTRAINING SET:")
print(f"Accuracy: {accuracy_score(y_train, mlp_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, mlp_train_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_train, mlp_train_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_train, mlp_train_pred, zero_division=0):.4f}")

print("\nVALIDATION SET:")
mlp_val_acc = accuracy_score(y_val, mlp_val_pred)
print(f"Accuracy: {mlp_val_acc:.4f}")
print(f"Precision: {precision_score(y_val, mlp_val_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val, mlp_val_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_val, mlp_val_pred, zero_division=0):.4f}")

print("\nTEST SET:")
mlp_test_acc = accuracy_score(y_test, mlp_test_pred)
print(f"Accuracy: {mlp_test_acc:.4f}")
print(f"Precision: {precision_score(y_test, mlp_test_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, mlp_test_pred, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(y_test, mlp_test_pred, zero_division=0):.4f}")

# Confusion Matrix
mlp_cm = confusion_matrix(y_test, mlp_test_pred)
print("\nConfusion Matrix:")
print(mlp_cm)

# Training history
print(f"\nMLP Training Details:")
print(f"Converged: {mlp_model.n_iter_} iterations")
print(f"Loss: {mlp_model.loss_:.4f}")

## 5. MLP Performance Visualization

In [ ]:
# MLP evaluation metrics
mlp_test_accuracy = accuracy_score(y_test, mlp_test_pred)
mlp_test_precision = precision_score(y_test, mlp_test_pred, zero_division=0)
mlp_test_recall = recall_score(y_test, mlp_test_pred, zero_division=0)
mlp_test_f1 = f1_score(y_test, mlp_test_pred, zero_division=0)

# Create metrics visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [mlp_test_accuracy, mlp_test_precision, mlp_test_recall, mlp_test_f1]
colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']

# Individual metric plots
for idx, (ax, metric, value, color) in enumerate(zip(axes.flat, metrics, values, colors)):
    ax.barh(['MLP'], [value], color=color, height=0.5)
    ax.set_xlim([0, 1])
    ax.set_xlabel('Score', fontsize=11)
    ax.set_title(f'MLP {metric} (Test Set)', fontsize=11, fontweight='bold')
    ax.text(value + 0.02, 0, f'{value:.4f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix visualization
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(mlp_cm, annot=True, fmt='d', cmap='Greens', ax=ax, cbar=False)
ax.set_title('MLP - Confusion Matrix (Test Set)', fontweight='bold', fontsize=12)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## 6. ROC Curve Analysis

In [ ]:
# MLP ROC Curve
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_test_proba)
roc_auc_mlp = auc(fpr_mlp, tpr_mlp)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_mlp, tpr_mlp, color='#2ca02c', lw=2.5, label=f'MLP (AUC = {roc_auc_mlp:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('MLP - ROC Curve (Test Set)', fontweight='bold', fontsize=12)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMLP AUC-ROC Score (Test Set): {roc_auc_mlp:.4f}")

In [ ]:
## 7. Export Trained Model

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('models/pca_mlp', exist_ok=True)

# Save MLP model using joblib
model_path_mlp = 'models/pca_mlp/mlp_model.pkl'

joblib.dump(mlp_model, model_path_mlp)

print("Model saved successfully!")
print(f"✓ MLP: {model_path_mlp}")

## 8. Summary Report

In [ ]:
# Summary Report
print("\n" + "="*60)
print("SUMMARY REPORT - PCA DATASET WITH MLP")
print("="*60)

print("\nTest Set Performance:")
print(f"{'Metric':<20} {'Score':<12}")
print("-" * 32)
print(f"{'Accuracy':<20} {mlp_test_accuracy:<12.4f}")
print(f"{'Precision':<20} {mlp_test_precision:<12.4f}")
print(f"{'Recall':<20} {mlp_test_recall:<12.4f}")
print(f"{'F1-Score':<20} {mlp_test_f1:<12.4f}")
print(f"{'AUC-ROC':<20} {roc_auc_mlp:<12.4f}")

print("\n" + "="*60)
print("MLP ARCHITECTURE DETAILS")
print("="*60)
print(f"Hidden Layers: {mlp_model.hidden_layer_sizes}")
print(f"Activation Function: {mlp_model.activation}")
print(f"Solver: {mlp_model.solver}")
print(f"Max Iterations: {mlp_model.max_iter}")
print(f"Learning Rate Init: {mlp_model.learning_rate_init}")
print(f"Actual Iterations to Convergence: {mlp_model.n_iter_}")
print(f"Final Loss: {mlp_model.loss_:.6f}")